# CC2017 – Deep Learning · Proyecto Final
## 02 · CNN entrenada desde cero

Se entrena una red convolucional propia sobre imágenes de 48×48. Esta es la línea base contra la que se compara el modelo preentrenado del notebook 03.

## Preparación · Librerías, reproducibilidad y dispositivo

In [ ]:
import sys, random, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
OUT = ROOT / 'outputs'; OUT.mkdir(exist_ok=True)

# FAST_DEV_RUN=True recorre pocos lotes por época para comprobar que todo corre; sus checkpoints llevan prefijo dev_.
FAST_DEV_RUN = False
MAX_BATCHES = 5 if FAST_DEV_RUN else None
PREFIX = 'dev_' if FAST_DEV_RUN else ''
NUM_WORKERS = 2
print('Dispositivo:', device, '| FAST_DEV_RUN:', FAST_DEV_RUN)

## 1. Datos

In [ ]:
from src.data import download_gtsrb, load_tables, split_train_val, GTSRBDataset, get_transforms, CLASS_NAMES, NUM_CLASSES

DATA_ROOT = download_gtsrb()
train_full, test_df = load_tables(DATA_ROOT)
train_df, val_df = split_train_val(train_full, val_size=0.15, seed=SEED)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
IMG_SIZE = 48
batch_size = 128

train_ds = GTSRBDataset(train_df, DATA_ROOT, get_transforms(IMG_SIZE, augment=True))
val_ds = GTSRBDataset(val_df, DATA_ROOT, get_transforms(IMG_SIZE))
train_loader = DataLoader(train_ds, batch_size, shuffle=True, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_ds, batch_size, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)

## 2. Arquitectura
Tres bloques `[Conv-BN-ReLU]×2 → MaxPool → Dropout2d` con 32, 64 y 128 filtros. Luego, pooling global promedio y un clasificador denso con dropout. El pooling global evita una capa densa grande y reduce parámetros.

In [ ]:
from src.models import TrafficCNN, count_params
cnn = TrafficCNN().to(device)
total, trainable = count_params(cnn)
print(cnn)
print(f'Parámetros: {total:,} (entrenables: {trainable:,})')

## 3. Entrenamiento
Adam con lr 1e-3, `ReduceLROnPlateau` y early stopping sobre la pérdida de validación. Se guardan los pesos de la mejor época. Si el checkpoint ya existe se carga en lugar de entrenar.

In [ ]:
epochs = 30
lr = 1e-3
optimizer = torch.optim.Adam(cnn.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)

from src.train import fit
t0 = time.time()
hist_cnn = fit(cnn, train_loader, val_loader, optimizer, device, epochs=2 if FAST_DEV_RUN else epochs,
               scheduler=scheduler, patience=5, ckpt_path=OUT / f'{PREFIX}cnn_scratch.pt', max_batches=MAX_BATCHES)
print(f'Tiempo total: {time.time() - t0:.0f}s')

## 4. Curvas de entrenamiento

In [ ]:
def plot_history(h, title):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    ep = range(1, len(h['train_loss']) + 1)
    axs[0].plot(ep, h['train_loss'], label='train'); axs[0].plot(ep, h['val_loss'], label='val'); axs[0].set(title='Pérdida', xlabel='Época')
    axs[1].plot(ep, h['train_acc'], label='train'); axs[1].plot(ep, h['val_acc'], label='val'); axs[1].set(title='Accuracy', xlabel='Época')
    for ax in axs: ax.legend(); ax.grid(alpha=.3)
    fig.suptitle(title); plt.tight_layout(); return fig

plot_history(hist_cnn, 'CNN desde cero').savefig(OUT / 'curvas_cnn.png', dpi=120); plt.show()
best = int(np.argmin(hist_cnn['val_loss']))
print(f"Mejor época: {best + 1} | val loss {hist_cnn['val_loss'][best]:.4f} | val acc {hist_cnn['val_acc'][best]:.4f}")

## 5. Métricas en validación

In [ ]:
from src.metrics import predict, summary_metrics
y_val, p_val, _ = predict(cnn, val_loader, device, MAX_BATCHES)
pd.DataFrame([summary_metrics(y_val, p_val, 'CNN desde cero (val)')]).round(4)

**Interpretación (completar después de ejecutar):** compare las curvas de train y val (¿hay sobreajuste? ¿en qué época?), indique la accuracy y F1 macro de validación, y registre el resultado en `docs/bitacora_iteraciones.md`. Recuerde que train lleva aumento de datos y dropout activos, por lo que su accuracy puede quedar por debajo de val.